# Step 6 — Drafting Agent + Grounding Verifier

Generates the three artifacts — **earnings-call script**, **deck outline**, **Q&A cheat sheet**
— with numbers **slotted** (`{{F-00xx}}`), then **verifies** every rendered value against the
Fact Store and runs a **compliance lint** (Safe Harbor + Reg-FD phrase check).

Key outputs from P1:
- **Safe Harbor** is the first script section (required for any IR call).
- **Metric labels are humanised** (no raw `gross_margin` keys in prose).
- **Compliance flags** surface selective-disclosure phrases (non-blocking; reviewer sees them).
- **Q&A cheat sheet** is grounded in the segment/YoY questions the Predictive Analyst produced.
- The **numeric verifier** blocks any ungrounded figure — demonstrated by injection.

In [1]:
import sys
from pathlib import Path
def _root():
    p = Path.cwd()
    for d in (p, *p.parents):
        if (d / "requirements.txt").exists():
            return d
    return p
ROOT = _root(); sys.path.insert(0, str(ROOT / "src"))
from ir_copilot.config import settings
print(f"ticker={settings.ticker}  period={settings.period}")
print(f"data={'mock(real cached)' if settings.use_mock_data else 'live defeatbeta-api'}  "
      f"embeddings={settings.embedding_backend}  sentiment={settings.sentiment_backend}  llm={settings.llm_backend}")

ticker=NVDA  period=FY2026Q2
data=mock(real cached)  embeddings=hash  sentiment=lexicon  llm=mock


In [2]:
from ir_copilot.facts import build_fact_store
from ir_copilot.embeddings import get_embedder
from ir_copilot.vectorstore import WikiStore, WikiChunk
from ir_copilot.corpus import wiki_chunks_for, news_items, signals
from ir_copilot.agents.sentiment import analyze_sentiment
from ir_copilot.agents.competitor import compare
from ir_copilot.agents.predictive import predict_questions
from ir_copilot.agents.drafting import draft, render_bundle
from ir_copilot.agents.verify import verify

store = build_fact_store(settings.ticker, settings.period, use_mock=settings.use_mock_data)
peers = {p: build_fact_store(p, settings.period, use_mock=settings.use_mock_data)
         for p in settings.peers}

tickers = [settings.ticker, *settings.peers]
wiki = WikiStore(get_embedder()); wiki.ensure_collection(recreate=True)
wiki.upsert([WikiChunk(chunk_id=str(i), **c)
             for i, c in enumerate(wiki_chunks_for(tickers))])

snap = analyze_sentiment(settings.ticker, news_items(settings.ticker))
pc   = compare(store, peers)
sigs = signals(settings.ticker)
qs   = predict_questions(store, snap, pc, wiki=wiki, signals=sigs)
bundle = draft(store, snap, pc, qs)
rendered = render_bundle(bundle, store)

In [3]:
# ── Display the full draft
print("=" * 68)
print("  EARNINGS CALL SCRIPT")
print("=" * 68)
for sec in rendered["script"]:
    print(f"\n## {sec['heading']}")
    print(sec['text'])

print("\n" + "=" * 68)
print("  INVESTOR DECK OUTLINE")
print("=" * 68)
for sl in rendered["deck_outline"]:
    print(f"\n### {sl['title']}")
    for b in sl["bullets"]: print(f"  • {b}")

print("\n" + "=" * 68)
print("  Q&A CHEAT SHEET  (predicted hard questions)")
print("=" * 68)
for i, qa in enumerate(rendered["qa_cheat_sheet"], 1):
    print(f"\n{i}. Q: {qa['question']}")
    print(f"   A: {qa['suggested_answer']}")
    if qa["evidence"]:
        print(f"   ↳ evidence: {qa['evidence'][0][:80]}")

  EARNINGS CALL SCRIPT

## Safe Harbor
Before we begin, a reminder that today's remarks contain forward-looking statements within the meaning of the Private Securities Litigation Reform Act of 1995. Actual results may differ materially due to risks described in our SEC filings. All figures are sourced from reported financials; we undertake no obligation to update forward-looking statements.

## Opening
Thank you for joining our FY2026Q2 earnings call. We delivered revenue of $81.61 billion with TTM EPS of $6.53, reflecting continued execution.

## Profitability
Gross margin was 74.9% and operating margin was 65.6%, supported by strong return on equity of 33.1% and ROIC of 31.4%.

## Competitive position
We continue to lead our peer group on gross margin, operating margin, ROE, ROIC, ROA, TTM EPS. Our market capitalization stands at $5.04 trillion.

  INVESTOR DECK OUTLINE

### Financial Highlights
  • Revenue $81.61 billion
  • TTM EPS $6.53
  • Gross margin 74.9%
  • Operating margin 

## Grounding + Compliance Verification

In [4]:
rep = verify(rendered, store)
print(f"Grounding passed    : {rep.passed}")
print(f"Numeric violations  : {rep.numeric_violations}")
print(f"Coverage missing    : {rep.coverage_missing}")
print(f"Compliance flags    : {rep.compliance_flags or 'none'}")
assert rep.passed,                   "clean draft must pass the grounding verifier"
assert not rep.numeric_violations,   "no ungrounded numbers"
assert any("safe harbor" in s["heading"].lower() for s in rendered["script"]),        "Safe Harbor section must be present"

# ── Inject a hallucinated number — verifier must catch it
import copy
bad = copy.deepcopy(rendered)
bad["qa_cheat_sheet"][0]["suggested_answer"] += " Our actual EPS guidance is 9.99 dollars."
rep_bad = verify(bad, store)
assert not rep_bad.passed and 9.99 in rep_bad.numeric_violations,        "hallucinated 9.99 must be caught"

print("\nGROUNDING ENFORCED:")
print("  ✓ Clean draft passes")
print("  ✓ Hallucinated 9.99 flagged:", rep_bad.numeric_violations)
print("  ✓ Safe Harbor section present")
print("  ✓ Compliance check ran")

Grounding passed    : True
Numeric violations  : []
Coverage missing    : []
Compliance flags    : none

GROUNDING ENFORCED:
  ✓ Clean draft passes
  ✓ Hallucinated 9.99 flagged: [9.99]
  ✓ Safe Harbor section present
  ✓ Compliance check ran


**Next (Step 7):** wire all agents into the LangGraph state machine with the human-in-the-loop
approval gate (`07_langgraph_orchestration.ipynb`).